In [1]:
import pandas as pd
df = pd.read_csv('data/titanic.csv')
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,0,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,1,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,0,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,0,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,1,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


The dataset contains the following columns:

- **PassengerId** - a unique identifier for each passenger
- **Survived** - indicates if the passenger survived the disaster (1) or not (0)
- **Pclass** - the class of the ticket (1st, 2nd or 3rd)
- **Name** - the name of the passenger
- **Sex** - the sex of the passenger
- **Age** - the age of the passenger
- **SibSp** - the number of siblings/spouses aboard the Titanic
- **Parch** - the number of parents/children aboard the Titanic
- **Ticket** - the ticket number
- **Fare** - the passenger fare
- **Cabin** - the cabin number
- **Embarked** - the port of embarkation (C = Cherbourg, Q = Queenstown, S = Southampton)

## Exercise 1: Data exploration and preprocessing (2 points)

1. **Check if there are any missing values in the dataset.** Print the percentage of missing values for each column.
2. **Draw distribution of the target column 'Survived' on a histogram.** Is the dataset balanced? What might be the problem with training a model on an imbalanced dataset?
3. **Delete columns that are unlikely to be useful for predicting the target column.** You can remove columns that contain unique identifiers, such as 'PassengerId', 'Name', and 'Ticket'. You can also remove the 'Cabin' column, as it contains many missing values.
4. **Check the size of the dataset before and after removing all missing values and print the results.** Is it really a good idea to remove all missing values from the dataset?

In [ ]:
# Your code goes here
...


Let's encode the 'Sex' column, which contain two categories 'male' and 'female'. We will use an approach called **label encoding**. The idea is to assign a unique integer $(0,1,2,3,...)$ to each category. Below is an example of label encoding for a column containing names of australian cities:

<center>
<img src="imgs/label-encoding.png" width=300>
</center>

In our case, we can assign $0$ to 'male' and $1$ to 'female', or the other way around. One simple way to do this is to create a dictionary with the mapping and use the `map` function from the `pandas` library.

In [ ]:
mapping = {'male': 0, 'female': 1}

df['Sex'] = df['Sex'].map(mapping)

df.head() # check the results

**This approach is simple and works well for some models, but it has a drawback**. Let's imagine we have a column of different australian cities and we use such a mapping to encode them:

```python
mapping = {"Albury": 0, 
           "Sydney": 1, 
           "Melbourne": 2,
           ...}

df['Location'] = df['Location'].map(mapping)
```

If we trained a predictive model on this data, we may face an issue. The model might deduce that 'Melbourne' is somehow more similar to 'Sydney' than to 'Albury', as 2 is closer to 1 than to 0. It may also expect that there is some order in the locations, which is probably not the case. Label encoding may be especially harmful for linear models, as we risk introducing an artificial order in the data.

To avoid those problems, we can use **one-hot encoding** (OHE). In this approach, we create a new binary column for each category. 

<center>
<img src="imgs/ohe-encoding.png" width=500>
</center>

All columns are independent, and the model will not assume any order in the data. The drawback is that the number of dimensions in the data increases significantly, which may lead to some problems related to [the curse of dimensionality](https://www.nature.com/articles/s41592-018-0019-x).

**We will use OHE strategy to encode the 'Embarked' column, which contains three categories: 'C', 'Q', and 'S'.** Although you may have an idea of how to implement one-hot encoding with some `pandas` methods and dataframe manipulations, OHE can be easily executed using the `get_dummies` function, as demonstrated below.

In [ ]:
locations_ohe = pd.get_dummies(df['Location'])
locations_ohe.head() # check the results

### *Implement one-hot encoding yourself, with the use of the `pandas` library.

If you are feeling confident with your `pandas` skills, you can try to implement one-hot encoding yourself.

The function `ohe_encode` should take a single argument, `column`, which is a pandas Series containing categorical data. The function should return a dataframe with one-hot encoded data, where the columns are named after the categories present in the input.

In [ ]:
def ohe_encode(column):
    # Your code goes here
    ...

## Dealing with missing values

As you have seen, dropping all the rows with missing values reduces the size of our dataset by over 20%. This is unacceptable, and we will circumvent this problem by using **imputation**.

Imputation is a process of replacing missing values with some estimated values. One of the simplest strategies is to replace missing values with the **mean** or the **median** of the column. You should be wary of using the mean, as it is sensitive to outliers. The median is more robust in this regard. The mean is generally a better choice for normally distributed data, while the median is better for skewed data.

In case of categorical data, you can replace missing values with the most frequent value in the column.

**Remember that you should calculate the mean or median on the training set and use the same values to impute missing values in the test set**. This is crucial, as you should not assume that you have access to the test set during the training phase, which would be an obvious case of **data leakage**.

## Exercise 2: Train-test split and fixing missing values (2 points)

- Split the dataframe into $X$ (features) and $y$ (target). The target column is 'Survived', and the features are all other columns (that we selected in the previous exercise).
- Split the features $X$ and labels $y$ into training and testing sets. Use 20% of the data for testing.
- Plot the distribution of 'Age' column. Impute missing values in the 'Age' column with either the mean or median of the training set.

In [ ]:
# Your code goes here
...

## Exercise 3: Featurizer class (2 points)

We did a lot of data preprocessing in the previous exercises. Let's encapsulate all the steps in a single class called `Featurizer`. This class will take care of encoding categorical columns, extracting labels, features, and imputing missing values. A *featurizer* is chunk of code which transforms raw input data into a processed form suitable for machine learning.

The `Featurizer` class should have the following methods:
- `fit(X)`: Calculate the median (or mean) of the 'Age' column and store it as a class attribute for imputation.
- `transform(X)`: Encode categorical columns, impute missing values, and return the resulting dataframe. If the `fit` method was not called before, the `transform` method should raise an exception. The method should also split the dataframe into features $X$ and labels $y$ and return them as a tuple.

In [ ]:
class Featurizer:
    # your code goes here
    ...

In [ ]:
# Test your implementation
from sklearn.model_selection import train_test_split

df = pd.read_csv('data/titanic.csv')
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

featurizer = Featurizer()
featurizer.fit(train_df)
train_X, train_y = featurizer.transform(train_df)
test_X, test_y = featurizer.transform(test_df)

train_X.head() # check the results